# Zep / Graphiti: temporal knowledge-graph memory

Graphiti builds a temporal knowledge graph from episodes and supports searching facts in context of when they were true. This notebook is a setup and experiment guide because the graph requires a database and configured model providers.

## Which memory type should I use?

Choose based on what must be remembered and what question the system must answer:

| Need | Memory type to choose | Typical fit |
|---|---|---|
| Continue the same conversation or workflow | Thread/checkpoint state | Recent messages, pending step, tool result |
| Personalize future conversations | Semantic/profile memory | Preferences, stable facts, goals |
| Recall a particular past event | Episodic memory | Decisions, meetings, dated interactions |
| Repeat a validated workflow | Procedural/experience memory | How-to steps, successful task lessons |
| Answer from reference documents | Document/RAG memory | Policies, manuals, knowledge bases |
| Connect people/things across facts | Entity/relationship graph | Multi-hop questions and linked entities |
| Answer what changed and when | Temporal memory | Versioned facts, evolving relationships |
| Keep large histories manageable | Hierarchical/consolidated memory | Profile summary plus recent episodes/archive |

Do not pick the most complex tool by default. Choose the simplest type that meets measured needs; many systems combine two types (for example, thread checkpoints plus a separate profile store). Retrieval must always be scoped to the authenticated user/tenant before results are sent to an LLM.

## Why this experiment uses this tool

**Primary type here:** Temporal entity/relationship memory.

**When to use it:** Questions about entities, relationships, and how facts change over time, including ?what was true when??

**Choose this tool when:** Choose Graphiti when temporal relationships are important enough to justify a graph database and extraction pipeline. For simple preferences, use profile memory; for independent source documents, use ordinary RAG.

**About this notebook:** The notebook is a setup and experiment guide. It requires a configured graph database and model providers; the configuration check does not write data.


## Setup requirements

Install the `graphiti-core` package, run a Neo4j instance, and configure the required LLM and embedding provider credentials through environment variables as described in Graphiti's official installation guide. Keep database credentials out of notebook cells. The next cell only checks that configuration is present; it does not connect or write data.

In [ ]:
%pip install -q graphiti-core

In [ ]:
import os

required = ("NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD", "OPENAI_API_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    print("Set these environment variables before connecting:", ", ".join(missing))
else:
    print("Configuration is present. Follow the installed Graphiti version's official quickstart to connect and add an episode.")

## Experiment design

Use the Graphiti quickstart matching your installed version to connect to the graph, add two dated episodes describing a changing preference, then query the current preference and the historical one. Compare results with ordinary vector retrieval: Graphiti's key distinction is entity relationships and temporal validity. Use a disposable database and a dedicated test group; clean up only that test data using the documented API.

## Ten cell-by-cell examples: Zep Graphiti

Run this section from top to bottom. It contains ten separately runnable, offline examples for common memory/RAG behaviors. The small lexical retriever below is only a deterministic baseline to explore inputs and metadata; it is **not** the project SDK or a semantic model. Use the project-specific cells above and the current official documentation to run these scenarios against the actual framework.

In [ ]:
import re


def offline_memory_search(records, query, user_id=None, top_k=3):
    """Simple token-overlap baseline; not semantic search or a framework API."""
    tokens = lambda text: set(re.findall(r"[a-z0-9]+", text.lower()))
    query_tokens = tokens(query)
    candidates = []
    for record in records:
        if user_id is not None and record.get("user_id") != user_id:
            continue
        if record.get("active", True) is False:
            continue
        overlap = query_tokens & tokens(record["text"])
        if overlap:
            candidates.append((len(overlap) / max(len(query_tokens), 1), record))
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[:top_k]


### Example 1: Profile recall

In [ ]:
records = [{'memory_id': 'memory-1', 'user_id': 'demo-user', 'text': 'The user is a data analyst who uses Python for reports.', 'source': 'synthetic-example'}]
query = 'What role does the user have?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 2: Preference recall

In [ ]:
records = [{'memory_id': 'memory-2', 'user_id': 'demo-user', 'text': 'The user prefers vegetarian meals and avoids dairy.', 'source': 'synthetic-example'}]
query = 'What vegetarian meals does the user prefer?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 3: Goal recall

In [ ]:
records = [{'memory_id': 'memory-3', 'user_id': 'demo-user', 'text': 'The user is training for a half marathon in October.', 'source': 'synthetic-example'}]
query = 'What half marathon is the user training for in October?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 4: Episodic recall

In [ ]:
records = [{'memory_id': 'memory-4', 'user_id': 'demo-user', 'text': '2026-04-12: The user visited Kyoto and enjoyed the botanical garden.', 'source': 'synthetic-example'}]
query = 'Which garden did the user enjoy in Kyoto?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 5: Updated fact

In [ ]:
records = [{'memory_id': 'memory-5-old', 'user_id': 'demo-user', 'text': 'The user prefers JavaScript examples.', 'active': False, 'source': 'synthetic-example'}, {'memory_id': 'memory-5-current', 'user_id': 'demo-user', 'text': 'The user currently prefers TypeScript examples.', 'active': True, 'source': 'synthetic-example'}]
query = 'What examples does the user currently prefer in TypeScript?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 6: User isolation

In [ ]:
records = [{'memory_id': 'memory-6', 'user_id': 'user-a', 'text': 'User A prefers concise answers. This record belongs only to user A.', 'source': 'synthetic-example'}, {'memory_id': 'memory-other-user', 'user_id': 'user-b', 'text': 'User B prefers detailed answers with citations.', 'source': 'synthetic-example'}]
query = 'What answer style does the user prefer?'
matches = offline_memory_search(records, query, user_id='user-a')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 7: Temporal context

In [ ]:
records = [{'memory_id': 'memory-7', 'user_id': 'demo-user', 'text': '2026-02-01: The project database was SQLite. 2026-06-01: The current project database is PostgreSQL after migration.', 'source': 'synthetic-example'}]
query = 'Which current project database is PostgreSQL after migration?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 8: Procedural memory

In [ ]:
records = [{'memory_id': 'memory-8', 'user_id': 'demo-user', 'text': 'When explaining code to this user, start with a short summary and then show a runnable example.', 'source': 'synthetic-example'}]
query = 'When explaining code to this user, what should come first and what should follow?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 9: Related facts

In [ ]:
records = [{'memory_id': 'memory-9', 'user_id': 'demo-user', 'text': "Mira works at Northstar Labs. Mira's company builds the Aurora assistant for researchers.", 'source': 'synthetic-example'}]
query = "Which assistant does Mira's company build?"
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")


### Example 10: Grounded RAG memory

In [ ]:
records = [{'memory_id': 'memory-10', 'user_id': 'demo-user', 'text': 'Source: handbook-section-4. Refund requests must be submitted within 30 days of purchase.', 'source': 'synthetic-example'}]
query = 'How many days do handbook refund requests have after purchase?'
matches = offline_memory_search(records, query, user_id='demo-user')
print("Query:", query)
print("Retrieved:", matches if matches else "No lexical matches; try a different query or inspect the memory text.")
